# PatchBench FR/EN : lancer les expériences sur Colab

**Avant de commencer** : Exécution > Modifier le type d'exécution > **GPU T4**.

Les résultats sont écrits sur ton **Google Drive** au fur et à mesure. Si la session coupe, relance simplement les cellules : le script reprend là où il s'est arrêté.

Ordre conseillé : 1) test smoke, 2) Qwen 3B, 3) Qwen 1.5B, 4) Llama 3B, 5) analyse.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Drive et code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/patchbench_results'
import os; os.makedirs(RESULTS, exist_ok=True)

In [ ]:
REPO_URL = 'https://github.com/<ton-pseudo>/patchbench.git'   # <- à remplacer
!rm -rf /content/patchbench && git clone -q $REPO_URL /content/patchbench
%cd /content/patchbench
!pip install -q -r requirements.txt

## 2. Accès à Llama 3.2 (seulement pour le modèle Llama)

Accepte la licence sur https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct, crée un token (Settings > Access Tokens, type Read), puis exécute la cellule. Tu peux sauter cette étape tant que tu fais tourner les modèles Qwen.

In [ ]:
from huggingface_hub import login
login()

## 3. Test smoke (environ 5 minutes)

Vérifie que tout marche et que le modèle respecte le format "Answer: <nombre>" / "Réponse : <nombre>". Les deux taux doivent être proches de 100 %.

In [ ]:
!python -m patchbench.run --model Qwen/Qwen2.5-1.5B-Instruct --preset smoke --batch_size 8 --out /content/smoke
import pandas as pd
s = pd.read_json('/content/smoke/Qwen2.5-1.5B-Instruct.jsonl', lines=True)
print(s.groupby('q_lang')[['marker_found', 'parsed', 'correct']].mean())
s[['exp', 'cond', 'ctx_lang', 'output', 'answer', 'correct']].sample(12, random_state=0)

## 4. Expériences principales

Chaque cellule affiche sa progression et une estimation du temps restant. En cas d'erreur de mémoire, le script réduit tout seul la taille des batchs.

In [ ]:
!python -m patchbench.run --model Qwen/Qwen2.5-3B-Instruct --preset standard --batch_size 4 --out $RESULTS

In [ ]:
!python -m patchbench.run --model Qwen/Qwen2.5-1.5B-Instruct --preset standard --batch_size 8 --out $RESULTS

In [ ]:
!python -m patchbench.run --model meta-llama/Llama-3.2-3B-Instruct --preset standard --batch_size 4 --out $RESULTS

## 5. Analyse : tous les tableaux et figures

In [ ]:
!python -m patchbench.analyze --results $RESULTS --out $RESULTS/analysis
from IPython.display import Image, display
display(Image(f'{RESULTS}/analysis/fig_position.png'))
display(Image(f'{RESULTS}/analysis/fig_decomposition.png'))

Pour m'envoyer les résultats : télécharge le dossier `patchbench_results` depuis ton Drive (clic droit > Télécharger), ou seulement les fichiers `.jsonl`.